In [1]:
!pip install optuna

In [2]:
# Importing the libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

In [3]:
# Prepare your data
from pathlib import Path

# Works in VS Code/local project; fallback keeps compatibility if path differs.
data_path = Path('data/cancer-risk-factors.csv')
if not data_path.exists():
    data_path = Path('../data/cancer-risk-factors.csv')

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path.resolve()}")
print(f"Shape: {df.shape}")

Loaded dataset from: C:\Users\Acer\Downloads\Cancer-Risk-Prediction-Complete_1\data\cancer-risk-factors.csv
Shape: (2000, 21)


In [4]:
df.head(5)

,Patient_ID,Cancer_Type,Age,Gender,Smoking,Alcohol_Use,Obesity,Family_History,Diet_Red_Meat,Diet_Salted_Processed,...,Physical_Activity,Air_Pollution,Occupational_Hazards,BRCA_Mutation,H_Pylori_Infection,Calcium_Intake,Overall_Risk_Score,BMI,Physical_Activity_Level,Risk_Level
0,LU0000,Breast,68,0,7,2,8,0,5,3,...,4,6,3,1,0,0,0.398696,28.0,5,Medium
1,LU0001,Prostate,74,1,8,9,8,0,0,3,...,1,3,3,0,0,5,0.424299,25.4,9,Medium
2,LU0002,Skin,55,1,7,10,7,0,3,3,...,1,8,10,0,0,6,0.605082,28.6,2,Medium
3,LU0003,Colon,61,0,6,2,2,0,6,2,...,6,4,8,0,0,8,0.318449,32.1,7,Low
4,LU0004,Lung,67,1,10,7,4,0,6,3,...,9,10,9,0,0,5,0.524358,25.1,2,Medium


In [5]:
X = df.drop(columns=['Risk_Level','Patient_ID', 'Cancer_Type'])

# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(df['Risk_Level'])

In [6]:
X

,Age,Gender,Smoking,Alcohol_Use,Obesity,Family_History,Diet_Red_Meat,Diet_Salted_Processed,Fruit_Veg_Intake,Physical_Activity,Air_Pollution,Occupational_Hazards,BRCA_Mutation,H_Pylori_Infection,Calcium_Intake,Overall_Risk_Score,BMI,Physical_Activity_Level
0,68,0,7,2,8,0,5,3,7,4,6,3,1,0,0,0.398696,28.0,5
1,74,1,8,9,8,0,0,3,7,1,3,3,0,0,5,0.424299,25.4,9
2,55,1,7,10,7,0,3,3,4,1,8,10,0,0,6,0.605082,28.6,2
3,61,0,6,2,2,0,6,2,4,6,4,8,0,0,8,0.318449,32.1,7
4,67,1,10,7,4,0,6,3,10,9,10,9,0,0,5,0.524358,25.1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,60,1,4,6,4,0,10,6,4,4,5,3,1,0,4,0.437539,30.3,3
1996,84,1,5,7,8,0,10,0,1,2,1,3,0,0,2,0.451128,25.9,4
1997,65,0,7,2,10,0,4,2,2,3,6,0,0,1,0,0.295760,22.5,3
1998,64,1,10,2,10,0,2,10,7,5,4,2,0,0,10,0.422201,25.3,3


In [7]:
y

array([2, 2, 2, ..., 1, 2, 2])

## why cancer_type was removed from the features list

- It's not a predictive input - it's an outcome label or categorical grouping that's already strongly correlated with Risk Level

- If we include it, the model will cheat by learning the mapping like Prostate --> High risk, instead of learning from true risk factors (smoking, BMI etc)

In [8]:
# Split into training/test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify=y)

In [9]:
# Feature Scaling

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
model = RandomForestClassifier(random_state=42, n_estimators=100)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [11]:
y_pred = model.predict(X_test)

print("Classification Report")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

Classification Report
              precision    recall  f1-score   support

        High       1.00      0.95      0.97        20
         Low       1.00      1.00      1.00        65
      Medium       1.00      1.00      1.00       315

    accuracy                           1.00       400
   macro avg       1.00      0.98      0.99       400
weighted avg       1.00      1.00      1.00       400

Confusion Matrix
[[ 19   0   1]
 [  0  65   0]
 [  0   0 315]]


The model accurately distinguishes all risk levels, with only 1 mistake out of 400 with imbalanced dataset.

## Logistic Regression

In [12]:
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)

LogisticRegression(random_state=42)

In [13]:
y_pred_lr = log_reg.predict(X_test)

print("Classification Report")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred_lr))

Classification Report
              precision    recall  f1-score   support

        High       1.00      0.80      0.89        20
         Low       0.97      0.95      0.96        65
      Medium       0.98      0.99      0.99       315

    accuracy                           0.98       400
   macro avg       0.98      0.92      0.95       400
weighted avg       0.98      0.98      0.98       400

Confusion Matrix
[[ 16   0   4]
 [  0  62   3]
 [  0   2 313]]


## Removing Overall_Risk_Score and retrying

In [14]:
X_new = df.drop(columns=['Risk_Level','Patient_ID', 'Cancer_Type', 'Overall_Risk_Score'])

# Encode the target variable
le_new = LabelEncoder()
y_new = le_new.fit_transform(df['Risk_Level'])

In [15]:
# Split into training/test

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size = 0.2, random_state=42, stratify=y)

In [16]:
# Feature Scaling

scaler_new = StandardScaler()
X_train_new = scaler_new.fit_transform(X_train_new)
X_test_new = scaler_new.transform(X_test_new)

In [17]:
model_new = RandomForestClassifier(random_state=42, n_estimators=100)
model_new.fit(X_train_new, y_train_new)

RandomForestClassifier(random_state=42)

In [18]:
y_pred_new = model_new.predict(X_test_new)

print("Classification Report")
print(classification_report(y_test_new, y_pred_new, target_names=le.classes_))

print("Confusion Matrix")
print(confusion_matrix(y_test_new, y_pred_new))

Classification Report
              precision    recall  f1-score   support

        High       1.00      0.05      0.10        20
         Low       0.85      0.34      0.48        65
      Medium       0.83      0.99      0.90       315

    accuracy                           0.83       400
   macro avg       0.89      0.46      0.49       400
weighted avg       0.84      0.83      0.80       400

Confusion Matrix
[[  1   0  19]
 [  0  22  43]
 [  0   4 311]]


In [19]:
df.Risk_Level.value_counts()

Risk_Level
Medium    1574
Low        324
High       102
Name: count, dtype: int64

## SMOTE - Synthetic Minority Over-Sampling Technique

In [20]:
X = df.drop(columns=['Risk_Level','Patient_ID', 'Cancer_Type', 'Overall_Risk_Score'])
y = df['Risk_Level']

In [21]:
# Split into training/test

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify=y)

In [22]:
from imblearn.over_sampling import SMOTE

In [23]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train,y_train)

print("Class Distribution")
print(y_train_res.value_counts())

Class Distribution
Risk_Level
Medium    1259
Low       1259
High      1259
Name: count, dtype: int64


In [24]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_res, y_train_res)

RandomForestClassifier(random_state=42)

In [25]:
y_pred = rf_model.predict(X_test)

print("Classification Report")
print(classification_report(y_test, y_pred))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

Classification Report
              precision    recall  f1-score   support

        High       0.41      0.35      0.38        20
         Low       0.73      0.58      0.65        65
      Medium       0.88      0.92      0.90       315

    accuracy                           0.84       400
   macro avg       0.67      0.62      0.64       400
weighted avg       0.83      0.84      0.83       400

Confusion Matrix
[[  7   0  13]
 [  0  38  27]
 [ 10  14 291]]


- Accuracy: 84%
- High Risk: 7 correctly classified, 13 mis-classified as Medium
- Low Risk: 38 correctly classified, 27 confused with Medium
- Medium Risk: 291 correctly classified, 24 confused as High/Low

- Still struggles to detect High-risk patients even with SMOTE

## Optuna Tuning

In [26]:
# Optuna tuning
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import classification_report, confusion_matrix, make_scorer, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

import numpy as np
import joblib  # optional, to save model/study

c:\Users\Acer\Downloads\Cancer-Risk-Prediction-Complete_1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
X_train_res.shape

(3777, 17)

In [28]:
from sklearn.metrics import f1_score, make_scorer

# Define macro F1 scorer explicitly for multiclass
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

scorer = make_scorer(macro_f1)


In [29]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'random_state': 42,
        'n_jobs': -1
    }

    model = RandomForestClassifier(**params)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train_res, y_train_res):
        X_t, X_v = X_train_res.iloc[train_idx], X_train_res.iloc[val_idx]
        y_t, y_v = y_train_res.iloc[train_idx], y_train_res.iloc[val_idx]

        model.fit(X_t, y_t)
        y_pred = model.predict(X_v)

        score = f1_score(y_v, y_pred, average='macro')  # explicit multiclass handling
        scores.append(score)

    return np.mean(scores)

In [30]:
# --- Create study ---
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, study_name='rf_macro_f1')


[I 2026-04-09 23:24:30,384] A new study created in memory with name: rf_macro_f1


In [31]:
# --- Run optimization ---
n_trials = 50  # change to 100+ if you want more thorough search
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

print("Best trial:")
print("  Value (macro F1):", study.best_value)
print("  Params:")
for k, v in study.best_params.items():
    print(f"    {k}: {v}")

Best trial: 0. Best value: 0.940236:   2%|▏         | 1/50 [00:02<01:57,  2.39s/it]

[I 2026-04-09 23:24:38,481] Trial 0 finished with value: 0.9402356457892479 and parameters: {'n_estimators': 218, 'max_depth': 29, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 0 with value: 0.9402356457892479.


Best trial: 1. Best value: 0.948868:   4%|▍         | 2/50 [00:03<01:06,  1.39s/it]

[I 2026-04-09 23:24:39,147] Trial 1 finished with value: 0.9488676606640238 and parameters: {'n_estimators': 59, 'max_depth': 30, 'min_samples_split': 9, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 1 with value: 0.9488676606640238.


Best trial: 1. Best value: 0.948868:   6%|▌         | 3/50 [00:05<01:27,  1.86s/it]

...

[I 2026-04-09 23:27:27,044] Trial 47 finished with value: 0.9528554570488716 and parameters: {'n_estimators': 321, 'max_depth': 16, 'min_samples_split': 8, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 41 with value: 0.9560722551368168.


Best trial: 41. Best value: 0.956072:  98%|█████████▊| 49/50 [02:55<00:03,  3.76s/it]

[I 2026-04-09 23:27:31,415] Trial 48 finished with value: 0.9442650744438706 and parameters: {'n_estimators': 480, 'max_depth': 22, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 41 with value: 0.9560722551368168.


Best trial: 41. Best value: 0.956072: 100%|██████████| 50/50 [02:58<00:00,  3.58s/it]

[I 2026-04-09 23:27:34,999] Trial 49 finished with value: 0.937829367661044 and parameters: {'n_estimators': 404, 'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 7, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 41 with value: 0.9560722551368168.
Best trial:
  Value (macro F1): 0.9560722551368168
  Params:
    n_estimators: 472
    max_depth: 15
    min_samples_split: 6
    min_samples_leaf: 2
    max_features: sqrt
    bootstrap: False
    criterion: entropy


In [32]:
# Train final model with best params and full training data, then evaluate on test
best_params = {
    'n_estimators': 347,
    'max_depth': 19,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt',
    'bootstrap': False,
    'criterion': 'gini',
    'random_state': 42,
    'n_jobs': -1
}


final_rf = RandomForestClassifier(**best_params)
final_rf.fit(X_train_res, y_train_res)         # resampled training set
y_pred_test = final_rf.predict(X_test)        # untouched test set

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred_test))
print(confusion_matrix(y_test, y_pred_test))


              precision    recall  f1-score   support

        High       0.36      0.25      0.29        20
         Low       0.72      0.58      0.64        65
      Medium       0.87      0.92      0.90       315

    accuracy                           0.83       400
   macro avg       0.65      0.59      0.61       400
weighted avg       0.82      0.83      0.83       400

[[  5   0  15]
 [  0  38  27]
 [  9  15 291]]


In [33]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, make_scorer, recall_score
import optuna
import numpy as np

# scorer (macro f1)
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

macro_f1_scorer = make_scorer(macro_f1)

def objective(trial):
    # RF params to try
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 6, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
        'max_features': trial.suggest_categorical('max_features', ['sqrt','log2']),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'criterion': trial.suggest_categorical('criterion', ['gini','entropy']),
        'random_state': 42,
        'n_jobs': 1   # avoid nested parallelism inside cross_val_score
    }

    sm = SMOTE(random_state=42)
    rf = RandomForestClassifier(**params)
    pipe = ImbPipeline([('smote', sm), ('rf', rf)])

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=macro_f1_scorer, n_jobs=1)
    return float(np.mean(scores))

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30)
print(study.best_value, study.best_params)


[I 2026-04-09 23:27:58,767] A new study created in memory with name: no-name-0fbe8546-1a36-4829-ab4d-33ee9dce57b7
[I 2026-04-09 23:28:03,176] Trial 0 finished with value: 0.671473898202647 and parameters: {'n_estimators': 212, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}. Best is trial 0 with value: 0.671473898202647.
[I 2026-04-09 23:28:05,152] Trial 1 finished with value: 0.6657242146375855 and parameters: {'n_estimators': 106, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 0 with value: 0.671473898202647.
[I 2026-04-09 23:28:09,287] Trial 2 finished with value: 0.6616028068735651 and parameters: {'n_estimators': 284, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': 'log2', 'bootstrap': False, 'criterion': 'gini'}. Best is trial 0 with value: 0.671473898202647.
[I 2026-04-0

0.696319735673656 {'n_estimators': 123, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'bootstrap': True, 'criterion': 'entropy'}


In [34]:
# Suppose best_params from study:
best = study.best_params
rf = RandomForestClassifier(**best, random_state=42, n_jobs=-1)

pipe_final = ImbPipeline([('smote', SMOTE(random_state=42)), ('rf', rf)])
pipe_final.fit(X_train, y_train)   # fit on original X_train (pipeline will SMOTE inside)

y_test_pred = pipe_final.predict(X_test)
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))


              precision    recall  f1-score   support

        High       0.33      0.40      0.36        20
         Low       0.70      0.62      0.66        65
      Medium       0.88      0.90      0.89       315

    accuracy                           0.82       400
   macro avg       0.64      0.64      0.64       400
weighted avg       0.83      0.82      0.83       400

[[  8   0  12]
 [  0  40  25]
 [ 16  17 282]]


## **XGBoost**

In [36]:
# baseline XGBoost (SMOTE on training split only)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

le = LabelEncoder()
le.fit(y_train)

y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42, n_jobs=-1)
smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train_enc)
xgb.fit(X_train_bal, y_train_bal)

y_pred_enc = xgb.predict(X_test)

y_pred = le.inverse_transform(y_pred_enc)
y_test_orig = le.inverse_transform(y_test_enc)

print("Baseline XGB Classifier")
print(classification_report(y_test_orig, y_pred))
print("Confusion Matrix")
print(confusion_matrix(y_test_orig, y_pred))

Baseline XGB Classifier
              precision    recall  f1-score   support

        High       0.39      0.45      0.42        20
         Low       0.79      0.68      0.73        65
      Medium       0.90      0.92      0.91       315

    accuracy                           0.85       400
   macro avg       0.69      0.68      0.68       400
weighted avg       0.86      0.85      0.85       400

Confusion Matrix
[[  9   0  11]
 [  0  44  21]
 [ 14  12 289]]


In [37]:
## Optuna + XGBoost that optimizes recall for the High class (SMOTE applied inside each fold).

# --- Imports ---
import optuna
from optuna.samplers import TPESampler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import numpy as np

# --- Settings ---
TARGET_LABEL = 'High'       # label whose recall we want to maximize
LABEL_ORDER = ['High', 'Low', 'Medium']  # must match your label names exactly
N_TRIALS = 40               # start with 40; increase to 100+ for more thorough search
CV_FOLDS = 3

# --- Encode labels once (keep mapping stable) ---
le = LabelEncoder()
le.fit(y_train)               # fit on training labels (or whole y if preferred)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)
# numeric index of target label in encoded space
target_index = int(np.where(le.classes_ == TARGET_LABEL)[0])

# --- Objective: maximize recall for TARGET_LABEL ---
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        # stable args
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'mlogloss'
    }

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    recalls = []

    for train_idx, val_idx in cv.split(X_train, y_train_enc):
        X_t, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_t, y_v = y_train_enc[train_idx], y_train_enc[val_idx]

        # Apply SMOTE only on the fold's training subset.
        smote = SMOTE(random_state=42)
        X_t_bal, y_t_bal = smote.fit_resample(X_t, y_t)

        # n_jobs=1 to avoid nested parallelism inside CV
        model = XGBClassifier(**params, n_jobs=1)
        model.fit(X_t_bal, y_t_bal)
        y_pred = model.predict(X_v)

        # compute recall per class in LABEL_ORDER and select target_index
        recs = recall_score(y_v, y_pred, labels=list(range(len(le.classes_))), average=None, zero_division=0)
        recall_high = recs[target_index]
        recalls.append(recall_high)

    return float(np.mean(recalls))

# --- Run Optuna study ---
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, study_name='xgb_high_recall')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\nBest CV mean High-recall:", study.best_value)
print("Best params:")
for k,v in study.best_params.items():
    print(f"  {k}: {v}")

# --- Train final model with best params and evaluate on test set ---
best_params = study.best_params.copy()
# ensure required fixed args for final fit
best_params.update({'use_label_encoder': False, 'eval_metric': 'mlogloss', 'random_state': 42})
final_xgb = XGBClassifier(**best_params, n_jobs=-1)

# Fit SMOTE only on training data, then train XGB.
smote_final = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote_final.fit_resample(X_train, y_train_enc)
final_xgb.fit(X_train_bal, y_train_bal)

# predict (encoded), then decode for readable report
y_test_pred_enc = final_xgb.predict(X_test)
y_test_pred = le.inverse_transform(y_test_pred_enc)
y_test_orig = y_test  # original readable labels

print("\nFINAL Test Classification Report (labels in original names):")
print(classification_report(y_test_orig, y_test_pred, labels=le.classes_))
print("Confusion Matrix (rows=true, cols=predicted, order = le.classes_):")
print(confusion_matrix(y_test_orig, y_test_pred, labels=le.classes_))


C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3961897717.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  target_index = int(np.where(le.classes_ == TARGET_LABEL)[0])
[I 2026-04-09 23:31:15,234] A new study created in memory with name: xgb_high_recall
  0%|          | 0/40 [00:00<?, ?it/s]C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3961897717.py:32: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),
Best trial: 0. Best value: 0.489418:   2%|▎         | 1/40 [00:01<00:53,  1.38s/it]C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3961897717.py:32: FutureWarning: suggest_logunifo

[I 2026-04-09 23:31:16,616] Trial 0 finished with value: 0.4894179894179895 and parameters: {'n_estimators': 181, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.4936111842654619, 'gamma': 0.7799726016810132, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 0.4894179894179895.


Best trial: 0. Best value: 0.489418:   5%|▌         | 2/40 [00:05<01:54,  3.01s/it]C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3961897717.py:32: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),


[I 2026-04-09 23:31:20,764] Trial 1 finished with value: 0.4532627865961199 and parameters: {'n_estimators': 260, 'max_depth': 10, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.899465584480253, 'gamma': 1.0616955533913808, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 0.4894179894179895.


Best trial: 2. Best value: 0.502646:   8%|▊         | 3/40 [00:07<01:31,  2.47s/it]C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3961897717.py:32: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),


...

[I 2026-04-09 23:32:15,271] Trial 37 finished with value: 0.6119929453262786 and parameters: {'n_estimators': 76, 'max_depth': 9, 'learning_rate': 0.06013918864441076, 'subsample': 0.7097072081372868, 'colsample_bytree': 0.586399442186891, 'gamma': 4.661230074006847, 'reg_alpha': 4.70704538538442, 'reg_lambda': 2.429281691922641}. Best is trial 7 with value: 0.6485890652557319.


Best trial: 7. Best value: 0.648589:  98%|█████████▊| 39/40 [01:00<00:00,  1.15it/s]C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3961897717.py:32: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.3),


[I 2026-04-09 23:32:16,125] Trial 38 finished with value: 0.6238977072310404 and parameters: {'n_estimators': 98, 'max_depth': 10, 'learning_rate': 0.10488693506565742, 'subsample': 0.7323172030968779, 'colsample_bytree': 0.6972822402840096, 'gamma': 4.595340531170809, 'reg_alpha': 2.980244102098565, 'reg_lambda': 1.8874068111119229}. Best is trial 7 with value: 0.6485890652557319.


Best trial: 7. Best value: 0.648589: 100%|██████████| 40/40 [01:02<00:00,  1.56s/it]


[I 2026-04-09 23:32:17,808] Trial 39 finished with value: 0.6234567901234568 and parameters: {'n_estimators': 197, 'max_depth': 12, 'learning_rate': 0.04322478701040103, 'subsample': 0.7452638432501364, 'colsample_bytree': 0.987907162839617, 'gamma': 3.518112587422709, 'reg_alpha': 4.251313643434011, 'reg_lambda': 2.5459838124191485}. Best is trial 7 with value: 0.6485890652557319.

Best CV mean High-recall: 0.6485890652557319
Best params:
  n_estimators: 81
  max_depth: 4
  learning_rate: 0.011662890273931383
  subsample: 0.7301321323053057
  colsample_bytree: 0.6332063738136893
  gamma: 1.3567451588694794
  reg_alpha: 4.143687545759647
  reg_lambda: 1.7837666334679465

FINAL Test Classification Report (labels in original names):
              precision    recall  f1-score   support

        High       0.16      0.60      0.26        20
         Low       0.50      0.77      0.60        65
      Medium       0.90      0.64      0.75       315

    accuracy                           0.

In [38]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 1. Encode target labels (if not already numeric)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# 2. Compute class weights inversely proportional to class frequencies
classes, counts = np.unique(y_train_enc, return_counts=True)
class_weights = {cls: max(counts)/count for cls, count in zip(classes, counts)}
print("Class Weights:", class_weights)

# 3. Initialize XGBoost with weights
xgb_weighted = XGBClassifier(
    objective='multi:softmax',
    num_class=len(classes),
    eval_metric='mlogloss',
    random_state=42,
    n_estimators=105,
    max_depth=3,
    learning_rate=0.014843806162322944,
    subsample=0.7730982444721965,
    colsample_bytree=0.8509237341976973,
    gamma=4.134336554829186,
    reg_alpha=4.453508368364819,
    reg_lambda=2.616923339470738,
    n_jobs=-1
)

# 4. Fit model with per-sample weights
sample_weights = np.array([class_weights[y] for y in y_train_enc])
xgb_weighted.fit(X_train, y_train_enc, sample_weight=sample_weights)

# 5. Evaluate (baseline before Optuna tuning)
y_pred_enc = xgb_weighted.predict(X_test)
print("Baseline Class-weighted XGBoost (fixed params) - Classification Report")
print(classification_report(y_test_enc, y_pred_enc, target_names=le.classes_))
print("Confusion Matrix")
print(confusion_matrix(y_test_enc, y_pred_enc))


Class Weights: {0: 15.353658536585366, 1: 4.861003861003861, 2: 1.0}
Baseline Class-weighted XGBoost (fixed params) - Classification Report
              precision    recall  f1-score   support

        High       0.21      0.80      0.34        20
         Low       0.42      0.80      0.55        65
      Medium       0.92      0.58      0.71       315

    accuracy                           0.63       400
   macro avg       0.51      0.73      0.53       400
weighted avg       0.80      0.63      0.67       400

Confusion Matrix
[[ 16   0   4]
 [  0  52  13]
 [ 59  73 183]]


In [44]:
# Optuna tuning for class-weighted XGBoost (optimize macro-F1)
import optuna
from optuna.samplers import TPESampler
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import numpy as np

# ----------------------
# Prepare encoded labels
# ----------------------
le = LabelEncoder()
le.fit(y_train)               # fit encoder on training labels
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)
n_classes = len(le.classes_)

# ----------------------
# Objective: maximize macro-F1
# ----------------------
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'max_depth': trial.suggest_int('max_depth', 2, 12),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
        # stable args:
        'use_label_encoder': False,
        'eval_metric': 'mlogloss',
        # set n_jobs=1 inside CV to avoid nested parallelism
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train_enc):
        X_t, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_t, y_v = y_train_enc[train_idx], y_train_enc[val_idx]

        # compute class weights from the fold's training data
        classes, counts = np.unique(y_t, return_counts=True)
        # simple inverse frequency scaling
        class_weight_map = {cls: float(max(counts) / cnt) for cls, cnt in zip(classes, counts)}
        sample_weights = np.array([class_weight_map[y] for y in y_t])

        # instantiate classifier with n_jobs=1 for CV
        model = XGBClassifier(**params, n_jobs=1, random_state=42)

        # fit with per-sample weights
        model.fit(X_t, y_t, sample_weight=sample_weights, verbose=False)

        y_pred = model.predict(X_v)
        scores.append(f1_score(y_v, y_pred, average='macro'))

    # return mean macro-F1 across folds
    return float(np.mean(scores))

# ----------------------
# Run Optuna study
# ----------------------
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, study_name='xgb_class_weighted_macroF1')

# fewer trials to start; increase to 50-150 when comfortable
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best CV macro-F1:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

# ----------------------
# Train final model on full training set using class weights
# ----------------------
best = study.best_params.copy()
best.update({'use_label_encoder': False, 'eval_metric': 'mlogloss', 'random_state': 42})

# compute final sample weights from full training set
classes, counts = np.unique(y_train_enc, return_counts=True)
final_class_weight_map = {cls: float(max(counts)/cnt) for cls, cnt in zip(classes, counts)}
final_sample_weights = np.array([final_class_weight_map[y] for y in y_train_enc])

final_xgb = XGBClassifier(**best, n_jobs=-1)
final_xgb.fit(X_train, y_train_enc, sample_weight=final_sample_weights, verbose=False)

# Evaluate on untouched test set
y_test_pred_enc = final_xgb.predict(X_test)
y_test_pred = le.inverse_transform(y_test_pred_enc)
y_test_orig = y_test  # already readable labels

print("\nFINAL Test Classification Report (class-weighted XGB):")
print(classification_report(y_test_orig, y_test_pred, labels=le.classes_))
print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test_orig, y_test_pred, labels=le.classes_))


[I 2026-04-09 23:37:09,510] A new study created in memory with name: xgb_class_weighted_macroF1
  0%|          | 0/40 [00:00<?, ?it/s]

C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3887491615.py:26: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
Best trial: 0. Best value: 0.708262:   2%|▎         | 1/40 [00:01<00:43,  1.11s/it]C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3887491615.py:26: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),


[I 2026-04-09 23:37:10,605] Trial 0 finished with value: 0.7082617917262457 and parameters: {'n_estimators': 181, 'max_depth': 12, 'learning_rate': 0.06504856968981275, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.4936111842654619, 'gamma': 0.7799726016810132, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 0.7082617917262457.


Best trial: 0. Best value: 0.708262:   5%|▌         | 2/40 [00:04<01:28,  2.34s/it]C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3887491615.py:26: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),


[I 2026-04-09 23:37:13,819] Trial 1 finished with value: 0.6540759814438694 and parameters: {'n_estimators': 260, 'max_depth': 9, 'learning_rate': 0.001124579825911934, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.899465584480253, 'gamma': 1.0616955533913808, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 0.7082617917262457.


...

[I 2026-04-09 23:38:00,380] Trial 37 finished with value: 0.7079511717903452 and parameters: {'n_estimators': 256, 'max_depth': 11, 'learning_rate': 0.13582223319946057, 'subsample': 0.7546800649864809, 'colsample_bytree': 0.7239805276342184, 'gamma': 0.88123387456458, 'reg_alpha': 3.3064091089796817, 'reg_lambda': 2.6332441284248995}. Best is trial 32 with value: 0.7223307017879933.


Best trial: 32. Best value: 0.722331:  98%|█████████▊| 39/40 [00:52<00:01,  1.43s/it]C:\Users\Acer\AppData\Local\Temp\ipykernel_22924\3887491615.py:26: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),


[I 2026-04-09 23:38:01,582] Trial 38 finished with value: 0.7004748393560715 and parameters: {'n_estimators': 299, 'max_depth': 8, 'learning_rate': 0.09235930731283669, 'subsample': 0.7822763706887431, 'colsample_bytree': 0.6036685072791683, 'gamma': 2.1604618811844425, 'reg_alpha': 0.2637800420157097, 'reg_lambda': 4.737939640285915}. Best is trial 32 with value: 0.7223307017879933.


Best trial: 32. Best value: 0.722331: 100%|██████████| 40/40 [00:53<00:00,  1.34s/it]


[I 2026-04-09 23:38:03,111] Trial 39 finished with value: 0.7181749203837389 and parameters: {'n_estimators': 342, 'max_depth': 10, 'learning_rate': 0.23669906019907733, 'subsample': 0.8140972951144148, 'colsample_bytree': 0.987907162839617, 'gamma': 0.3884593998243786, 'reg_alpha': 4.251313643434011, 'reg_lambda': 1.7335263513565262}. Best is trial 32 with value: 0.7223307017879933.
Best CV macro-F1: 0.7223307017879933
Best params:
  n_estimators: 272
  max_depth: 12
  learning_rate: 0.0891879979030594
  subsample: 0.7012866277412377
  colsample_bytree: 0.5316068128421201
  gamma: 1.068501671883885
  reg_alpha: 0.35571336905345036
  reg_lambda: 4.9617905262303115

FINAL Test Classification Report (class-weighted XGB):
              precision    recall  f1-score   support

        High       0.53      0.45      0.49        20
         Low       0.71      0.77      0.74        65
      Medium       0.92      0.91      0.91       315

    accuracy                           0.86       400

In [45]:
from sklearn.metrics import accuracy_score

# Recompute from the final class-weighted XGBoost model to avoid stale in-memory variables.
if all(name in globals() for name in ['final_xgb', 'X_test', 'y_test_enc']):
    y_pred_final = final_xgb.predict(X_test)
    final_acc = accuracy_score(y_test_enc, y_pred_final)
    print(f"Latest final accuracy (recomputed from final_xgb): {final_acc:.2f}")
else:
    print("Final model/test variables are not in memory. Run the final class-weighted XGBoost cell first.")

Latest final accuracy (recomputed from final_xgb): 0.86


## Winner: Optuna-Tuned Class-weighted XGBoost

![Final class-weighted XGBoost accuracy 0.86](assets/final_accuracy_086.png)

Latest verified final test accuracy from the current run: **0.86**

## Save the model as a .pkl file

In [41]:
import joblib

joblib.dump(model,'model_xgb_new.pkl')

print("Model saved successfully as 'model_xgb_new.pkl'")

Model saved successfully as 'model_xgb_new.pkl'
